task-importance structure conditional on independently classified GenAI affordance mechanism

It Measures Task Importance Distribution Using SD, IQR

In [ ]:
# ------------------------------------------------------------------
# Step 1 — Load inputs
# ------------------------------------------------------------------

from pathlib import Path
import pandas as pd
import numpy as np

BASE_DIR = Path(
    "/content/drive/MyDrive/GenAI-task-affordance-labor-outcomes-full"
)

LAYER1_FILE = (
    BASE_DIR
    / "03_Layer1_Task_Affordance_Count"
    / "Output"
    / "Occupational_Task_Affordance_Layer1_Prevalence_v6.csv"
)

LAYER2_FILE = (
    BASE_DIR
    / "04_Layer2_Task_Affordance_Weighted"
    / "Output"
    / "Occupational_Task_Affordance_Layer2_Salience_v6.csv"
)

CLASSIFICATION_FILE = (
    BASE_DIR
    / "02_Task_Data_and_Classification"
    / "Output"
    / "Occupational_Task_Classification_Provisional_v6.csv"
)

layer1 = pd.read_csv(LAYER1_FILE)
layer2 = pd.read_csv(LAYER2_FILE)
tasks = pd.read_csv(CLASSIFICATION_FILE)

In [ ]:
# ------------------------------------------------------------------
# Step 2 — Create the Layer 3 task universe
# ------------------------------------------------------------------

MECHANISM_LABELS = [
    "Automation-oriented",
    "Augmentation-oriented",
    "Limited GenAI applicability",
]

layer3_tasks = tasks.loc[
    tasks["provisional_label"].isin(MECHANISM_LABELS)
].copy()

assert len(layer3_tasks) == 16_445
assert layer3_tasks["soc6"].nunique() == 771

print(f"Layer 3 task records: {len(layer3_tasks):,}")
print(f"SOC6 occupations: {layer3_tasks['soc6'].nunique():,}")

In [ ]:
# ------------------------------------------------------------------
# Step 3 — Construct effective Layer 3 task importance
# ------------------------------------------------------------------

source_mean = (
    layer3_tasks.groupby("onet_soc_code")["task_importance"]
    .transform("mean")
)

soc6_mean = (
    layer3_tasks.groupby("soc6")["task_importance"]
    .transform("mean")
)

layer3_tasks["task_importance_effective_l3"] = (
    layer3_tasks["task_importance"]
    .fillna(source_mean)
    .fillna(soc6_mean)
)

In [ ]:
# ------------------------------------------------------------------
# Step 4 — Calculate within-SOC6 importance positions
# ------------------------------------------------------------------

occupation_task_count = (
    layer3_tasks.groupby("soc6")["task_id"]
    .transform("size")
)

layer3_tasks["importance_midrank_l3"] = (
    layer3_tasks.groupby("soc6")["task_importance_effective_l3"]
    .rank(method="average", ascending=True)
)

layer3_tasks["importance_position_l3"] = (
    layer3_tasks["importance_midrank_l3"] - 0.5
) / occupation_task_count

Layer 3 was left undefined for SOC6 occupations lacking any observed task-importance ratings because no within-occupation importance ordering could be estimated.

In [ ]:
# ------------------------------------------------------------------
# Step 5 — Calculate within-SOC6 importance positions
# ------------------------------------------------------------------

occupation_task_count = (
    layer3_tasks.groupby("soc6")["task_id"]
    .transform("size")
)

layer3_tasks["importance_midrank_l3"] = (
    layer3_tasks.groupby("soc6")["task_importance_effective_l3"]
    .rank(method="average", ascending=True)
)

layer3_tasks["importance_position_l3"] = (
    layer3_tasks["importance_midrank_l3"] - 0.5
) / occupation_task_count

In [ ]:
# ------------------------------------------------------------------
# Step 6 — Calculate Layer 3 mechanism positions
# ------------------------------------------------------------------

layer3_positions = (
    layer3_tasks.groupby(
        ["soc6", "provisional_label"]
    )["importance_position_l3"]
    .mean()
    .unstack()
    .reindex(columns=MECHANISM_LABELS)
    .rename(columns={
        "Automation-oriented": "automation_position",
        "Augmentation-oriented": "augmentation_position",
        "Limited GenAI applicability": "limited_position",
    })
    .reset_index()
)
layer3_profiles = (
    layer3_tasks[["soc6"]]
    .drop_duplicates()
    .merge(
        layer3_positions,
        on="soc6",
        how="left",
        validate="one_to_one",
    )
)

In [ ]:
# ------------------------------------------------------------------
# Step 7 — Validate Layer 3 construction
# ------------------------------------------------------------------

occupation_mean_position = (
    layer3_tasks.groupby("soc6")["importance_position_l3"]
    .mean()
)

assert np.allclose(
    occupation_mean_position.dropna(),
    0.5,
    atol=1e-12,
)

In [ ]:
# ------------------------------------------------------------------
# Step 8 — Assemble the occupational configuration file
# ------------------------------------------------------------------

configuration = (
    layer1
    .merge(
        layer2,
        on="soc6",
        how="left",
        validate="one_to_one",
    )
    .merge(
        layer3_profiles,
        on="soc6",
        how="left",
        validate="one_to_one",
    )
)

assert len(configuration) == 771
assert configuration["soc6"].is_unique

print(f"Configuration records: {len(configuration):,}")
print(f"SOC6 occupations: {configuration['soc6'].nunique():,}")

In [ ]:
# ------------------------------------------------------------
# Document Layer 3 position eligibility and missingness
# ------------------------------------------------------------

layer3_eligibility = (
    layer3_tasks
    .groupby("soc6")
    .agg(
        observed_importance_count=("task_importance", "count"),
        effective_importance_count=(
            "task_importance_effective_l3",
            "count"
        )
    )
    .reset_index()
)

layer3_eligibility["layer3_position_eligible"] = (
    layer3_eligibility["effective_importance_count"] > 0
)

layer3_eligibility["layer3_missing_reason"] = pd.NA
layer3_eligibility.loc[
    ~layer3_eligibility["layer3_position_eligible"],
    "layer3_missing_reason"
] = "no_onet_importance_records"

configuration = configuration.merge(
    layer3_eligibility,
    on="soc6",
    how="left",
    validate="one_to_one"
)

for mechanism in ["automation", "augmentation", "limited"]:
    count_column = f"{mechanism}_count"
    position_column = f"{mechanism}_position"
    status_column = f"{mechanism}_position_status"

    configuration[status_column] = "observed"

    configuration.loc[
        configuration[count_column].eq(0),
        status_column
    ] = "mechanism_absent"

    configuration.loc[
        configuration[count_column].gt(0)
        & ~configuration["layer3_position_eligible"],
        status_column
    ] = "no_onet_importance_records"

    unexpected_missing = (
        configuration[count_column].gt(0)
        & configuration["layer3_position_eligible"]
        & configuration[position_column].isna()
    )

    configuration.loc[
        unexpected_missing,
        status_column
    ] = "unexpected_missing"

    assert not unexpected_missing.any()
    assert not (
        configuration[count_column].eq(0)
        & configuration[position_column].notna()
    ).any()

position_status_audit = pd.concat(
    {
        mechanism: configuration[
            f"{mechanism}_position_status"
        ].value_counts()
        for mechanism in ["automation", "augmentation", "limited"]
    },
    axis=1
).fillna(0).astype(int)

print(
    "Layer 3 position-eligible occupations:",
    configuration["layer3_position_eligible"].sum()
)

display(position_status_audit)

In [ ]:
# ------------------------------------------------------------------
# Step 8 — Save final occupational configuration
# ------------------------------------------------------------------

OUTPUT_FILE = (
    BASE_DIR
    / "05_Layer3_Task_Configuration"
    / "Output"
    / "Occupational_Task_Affordance_Configuration_v6.csv"
)
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)

configuration.to_csv(OUTPUT_FILE, index=False)

print(f"Saved: {OUTPUT_FILE}")
print(f"Rows: {len(configuration):,}")

# AUDIT PURPOSE

In [ ]:
tasks_audit = pd.read_csv(CLASSIFICATION_FILE)

In [ ]:
print(CLASSIFICATION_FILE)
print(f"Tasks: {len(tasks_audit):,} | SOC6: {tasks_audit['soc6'].nunique():,}")

In [ ]:
# Substantive Layer 1–3 task universe
audit_tasks = tasks_audit.loc[
    tasks_audit["provisional_label"].isin(MECHANISM_LABELS)
].copy()

assert len(audit_tasks) == 16_445
assert audit_tasks["soc6"].nunique() == 771

# Normalize statements for exact cross-code comparison
audit_tasks["statement_key"] = (
    audit_tasks["task_statement"]
    .astype("string")
    .str.strip()
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
)

# SOC6 occupations receiving tasks from multiple O*NET codes
multi_code_soc6 = (
    audit_tasks.groupby("soc6")["onet_soc_code"]
    .nunique()
    .loc[lambda x: x > 1]
    .index
)

# Identical statements appearing across different O*NET codes
cross_code_repeats = (
    audit_tasks.loc[audit_tasks["soc6"].isin(multi_code_soc6)]
    .groupby(["soc6", "statement_key"])
    .agg(
        record_count=("task_id", "size"),
        contributing_codes=("onet_soc_code", "nunique"),
    )
    .reset_index()
    .loc[lambda x: x["contributing_codes"] > 1]
)

print(f"Multi-code SOC6 occupations: {len(multi_code_soc6)}")
print(f"Repeated cross-code statements: {len(cross_code_repeats)}")
print(
    "SOC6 occupations affected:",
    cross_code_repeats["soc6"].nunique(),
)

In [ ]:
repeat_records = (
    audit_tasks.merge(
        cross_code_repeats[["soc6", "statement_key"]],
        on=["soc6", "statement_key"],
        how="inner",
        validate="many_to_one",
    )
    .sort_values(
        ["soc6", "statement_key", "onet_soc_code", "task_id"]
    )
)

display(
    repeat_records[
        [
            "soc6",
            "onet_soc_code",
            "occupation_title",
            "task_source_type",
            "task_id",
            "task_statement",
            "provisional_label",
            "task_type",
            "task_importance",
        ]
    ]
)